In [1]:
import os
import json
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm


def rcis2_to_class(rcis2: float) -> int:
    """
    Convert continuous RCI_S2 into 3 classes.

    Class mapping:
        0 = flexible         (RCI_S2 <= 0.69)
        1 = context-dependent (0.69 < RCI_S2 < 0.80)
        2 = rigid            (RCI_S2 >= 0.80)
    """
    if rcis2 <= 0.69:
        return 0
    elif rcis2 >= 0.80:
        return 2
    else:
        return 1


def build_dataset(
    embedding_dir,
    residue_csv,
    output_dir,
    save_metadata_csv=True,
):
    os.makedirs(output_dir, exist_ok=True)

    df = pd.read_csv(residue_csv)

    required_cols = ["protein_id", "residue_order_0", "RCI_S2"]
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns in residue CSV: {missing_cols}")

    X = []
    y_class = []
    y_rcis2 = []
    groups = []
    metadata_rows = []

    missing_embeddings = 0
    length_mismatches = 0
    out_of_bounds = 0
    used_residues = 0

    for protein_id in tqdm(df["protein_id"].unique(), desc="Processing proteins"):
        emb_path = os.path.join(embedding_dir, f"{protein_id}.pt")

        if not os.path.exists(emb_path):
            print(f"[WARNING] Missing embedding: {protein_id}")
            missing_embeddings += 1
            continue

        data = torch.load(emb_path, map_location="cpu")

        if "embedding" not in data or "sequence" not in data:
            print(f"[WARNING] Invalid embedding file format: {protein_id}")
            continue

        embedding = data["embedding"]
        sequence = data["sequence"]

        if isinstance(embedding, torch.Tensor):
            embedding = embedding.numpy()

        protein_df = df[df["protein_id"] == protein_id].copy()

        # Safety check: sequence length must match embedding rows
        if len(sequence) != embedding.shape[0]:
            print(
                f"[ERROR] Length mismatch for {protein_id}: "
                f"len(sequence)={len(sequence)} vs embedding.shape[0]={embedding.shape[0]}"
            )
            length_mismatches += 1
            continue

        for _, row in protein_df.iterrows():
            idx = int(row["residue_order_0"])

            if idx < 0 or idx >= embedding.shape[0]:
                print(f"[WARNING] Index out of bounds: {protein_id}, idx={idx}")
                out_of_bounds += 1
                continue

            rcis2 = float(row["RCI_S2"])
            cls = rcis2_to_class(rcis2)

            X.append(embedding[idx])
            y_class.append(cls)
            y_rcis2.append(rcis2)
            groups.append(protein_id)

            metadata = {
                "protein_id": protein_id,
                "residue_order_0": idx,
                "RCI_S2": rcis2,
                "class_label": cls,
            }

            # Optional extra metadata if available in residue CSV
            for optional_col in ["residue_name", "residue", "aa", "secondary_structure"]:
                if optional_col in row.index:
                    metadata[optional_col] = row[optional_col]

            metadata_rows.append(metadata)
            used_residues += 1

    X = np.asarray(X, dtype=np.float32)
    y_class = np.asarray(y_class, dtype=np.int64)
    y_rcis2 = np.asarray(y_rcis2, dtype=np.float32)
    groups = np.asarray(groups)

    print("\nFinal dataset summary:")
    print(f"X shape: {X.shape}")
    print(f"y_class shape: {y_class.shape}")
    print(f"y_rcis2 shape: {y_rcis2.shape}")
    print(f"groups shape: {groups.shape}")
    print(f"Used residues: {used_residues}")
    print(f"Missing embeddings: {missing_embeddings}")
    print(f"Length mismatches: {length_mismatches}")
    print(f"Out-of-bounds residues: {out_of_bounds}")

    # Class distribution
    unique, counts = np.unique(y_class, return_counts=True)
    class_names = {0: "flexible", 1: "context-dependent", 2: "rigid"}
    print("\nClass distribution:")
    for u, c in zip(unique, counts):
        print(f"  {u} ({class_names[u]}): {c}")

    # Save arrays
    np.save(os.path.join(output_dir, "X.npy"), X)
    np.save(os.path.join(output_dir, "y_class.npy"), y_class)
    np.save(os.path.join(output_dir, "y_rcis2.npy"), y_rcis2)
    np.save(os.path.join(output_dir, "groups.npy"), groups)

    # Save metadata CSV for later notebook analysis
    if save_metadata_csv:
        metadata_df = pd.DataFrame(metadata_rows)
        metadata_df.to_csv(os.path.join(output_dir, "metadata.csv"), index=False)

    # Save label mapping
    label_mapping = {
        "0": "flexible (RCI_S2 <= 0.69)",
        "1": "context-dependent (0.69 < RCI_S2 < 0.80)",
        "2": "rigid (RCI_S2 >= 0.80)",
    }
    with open(os.path.join(output_dir, "label_mapping.json"), "w") as f:
        json.dump(label_mapping, f, indent=2)

    print("\n✅ Dataset saved.")
    print(f"Saved to: {output_dir}")

In [2]:
build_dataset(
    embedding_dir="/home/akshay-paliwal/Documents/VUB/thesis_ai/embeddings/esm2",
    residue_csv="/home/akshay-paliwal/Documents/VUB/thesis_ai/data/processed_data/residue_data.csv",
    output_dir="/home/akshay-paliwal/Documents/VUB/thesis_ai/data/dataset/esm2_multiclass",
)


Processing proteins: 100%|██████████| 4089/4089 [02:53<00:00, 23.53it/s]



Final dataset summary:
X shape: (444304, 1280)
y_class shape: (444304,)
y_rcis2 shape: (444304,)
groups shape: (444304,)
Used residues: 444304
Missing embeddings: 0
Length mismatches: 0
Out-of-bounds residues: 0

Class distribution:
  0 (flexible): 94773
  1 (context-dependent): 80953
  2 (rigid): 268578

✅ Dataset saved.
Saved to: /home/akshay-paliwal/Documents/VUB/thesis_ai/data/dataset/esm2_multiclass


In [3]:
build_dataset(
    embedding_dir="/home/akshay-paliwal/Documents/VUB/thesis_ai/embeddings/protT5",
    residue_csv="/home/akshay-paliwal/Documents/VUB/thesis_ai/data/processed_data/residue_data.csv",
    output_dir="/home/akshay-paliwal/Documents/VUB/thesis_ai/data/dataset/protT5_multiclass",
)

Processing proteins: 100%|██████████| 4089/4089 [02:28<00:00, 27.52it/s]



Final dataset summary:
X shape: (444304, 1024)
y_class shape: (444304,)
y_rcis2 shape: (444304,)
groups shape: (444304,)
Used residues: 444304
Missing embeddings: 0
Length mismatches: 0
Out-of-bounds residues: 0

Class distribution:
  0 (flexible): 94773
  1 (context-dependent): 80953
  2 (rigid): 268578

✅ Dataset saved.
Saved to: /home/akshay-paliwal/Documents/VUB/thesis_ai/data/dataset/protT5_multiclass


In [2]:
build_dataset(
    embedding_dir="/home/akshay-paliwal/Documents/VUB/thesis_ai/embeddings/esm3",
    residue_csv="/home/akshay-paliwal/Documents/VUB/thesis_ai/data/processed_data/residue_data.csv",
    output_dir="/home/akshay-paliwal/Documents/VUB/thesis_ai/data/dataset/esm3_multiclass",
)

Processing proteins: 100%|██████████| 4089/4089 [01:31<00:00, 44.56it/s]



Final dataset summary:
X shape: (444304, 1536)
y_class shape: (444304,)
y_rcis2 shape: (444304,)
groups shape: (444304,)
Used residues: 444304
Missing embeddings: 0
Length mismatches: 0
Out-of-bounds residues: 0

Class distribution:
  0 (flexible): 94773
  1 (context-dependent): 80953
  2 (rigid): 268578

✅ Dataset saved.
Saved to: /home/akshay-paliwal/Documents/VUB/thesis_ai/data/dataset/esm3_multiclass
